# 08 — Altitude Selection

**Route:** C81 (Campbell Airport, Grayslake, IL) -> KDLH (Duluth International, MN)

Picks a VFR cruising altitude for the route, constrained by:
- **Terrain/obstacles** -- a Maximum-Elevation-Figure-style floor (`vfr.terrain`), computed along the actual route rather than a chart's coarse 30-minute quadrangle.
- **Controlled airspace** -- Class B/C/D shelves the route passes under (`vfr.airspace`), from the FAA's own Class Airspace shapefile (real polygons, not an approximated circle).
- **Weather** -- freezing level (icing), forecast ceiling/visibility, and SIGMET/AIRMET hazards (`vfr.weather`), all live from aviationweather.gov.
- **Aircraft performance** -- a configurable service-ceiling profile (`vfr.aircraft`), not hardcoded to one airplane.

The final number is rounded to a legal VFR hemispheric cruising altitude per FAR 91.159 (odd thousands +500ft for 0-179 degree course, even thousands +500ft for 180-359 degree course). Ceiling/visibility and hazards are reported separately as a go/no-go readout -- those are about *whether* to fly, not *what altitude*.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from vfr import aircraft, airports, airspace, altitude, geo, magnetic, terrain, weather

FAA_CACHE_DIR = PROJECT_ROOT / "data" / "raw" / "faa_nasr"

## Step 1 — Route and aircraft

Same C81->KDLH route as Notebook 01. `AIRCRAFT_PROFILE` is a name resolved
against `data/aircraft/<name>.json` -- swap it for any other aircraft's
profile without touching code.

In [2]:
AIRCRAFT_PROFILE = "c172"

dep = airports.get_airport("C81")
dest = airports.get_airport("KDLH")
route_start = (dep["lat"], dep["lon"])
route_end = (dest["lat"], dest["lon"])
route_bearing_deg = geo.bearing_deg(dep["lat"], dep["lon"], dest["lat"], dest["lon"])

profile = aircraft.load_aircraft_profile(AIRCRAFT_PROFILE)
print(f"{dep['ident']} -> {dest['ident']}, true course {route_bearing_deg:.0f} deg")
print(f"Aircraft: {profile['type']}, service ceiling {profile['service_ceiling_ft']} ft")

KC81 -> KDLH, true course 328 deg
Aircraft: Cessna 172, service ceiling 13500 ft


## Step 2 — Terrain/obstacle floor

MEF-style: highest of (terrain + 300ft) or (registered obstacle top +
100ft) along the route, rounded up to the next 100ft -- see
`vfr.terrain` for the FAA margin rule this mirrors.

In [3]:
floor_ft = terrain.min_safe_altitude_msl(route_start, route_end, faa_cache_dir=FAA_CACHE_DIR)
print(f"Terrain/obstacle floor: {floor_ft:.0f} ft MSL")

Terrain/obstacle floor: 2200 ft MSL


## Step 3 — Airspace ceiling

Highest altitude that stays under every Class B/C/D shelf the route
passes through (excluding the departure/destination airport's own Class
D, since landing there is expected, not a routing conflict). `None`
means the route doesn\'t cross any controlled airspace at all.

In [4]:
shp_path = airspace.ensure_class_airspace_shapefile(FAA_CACHE_DIR)
airspace_ceiling_ft = airspace.max_airspace_altitude_msl(route_start, route_end, shp_path)
print(f"Airspace ceiling: {airspace_ceiling_ft} ft MSL")

Airspace ceiling: 3600.0 ft MSL


## Step 4 — Weather

Freezing level (icing avoidance) from the winds/temps-aloft forecast,
sampled near the route midpoint. Ceiling/visibility and SIGMET/AIRMET
hazards are live current-conditions data -- reported separately below as
a go/no-go readout, not folded into the altitude number.

In [5]:
mid_lat = (route_start[0] + route_end[0]) / 2
mid_lon = (route_start[1] + route_end[1]) / 2

freezing = weather.freezing_level(mid_lat, mid_lon)
freezing_level_ft = freezing["ft"] if freezing else None
if freezing is None:
    print("Freezing level: above forecast range (no icing concern)")
else:
    print(f"Freezing level: {'at or below ' if freezing['at_or_below'] else ''}{freezing_level_ft:.0f} ft MSL")

cv = weather.ceiling_visibility_along_route(route_start, route_end)
print(f"Lowest ceiling near route: {cv['min_ceiling_ft']} ft AGL")
print(f"Lowest visibility near route: {cv['min_visibility_sm']} SM")

hazards = weather.hazards_along_route(route_start, route_end)
print(f"SIGMETs/AIRMETs intersecting route: {len(hazards)}")
for h in hazards:
    print(f"  {h['hazard']} ({h['type']}), {h['altitude_low_ft']}-{h['altitude_high_ft']} ft")

Freezing level: 13714 ft MSL


Lowest ceiling near route: 100 ft AGL
Lowest visibility near route: 0.5 SM
SIGMETs/AIRMETs intersecting route: 0


## Step 5 — Combine into a recommendation

Valid band = `[terrain_floor, min(airspace_ceiling, aircraft_ceiling)]`.
The freezing level is not part of it: icing needs visible moisture as well
as cold, so capping the band there forbade clear winter air and said
nothing about cloud -- `vfr.altitude` reports it as a warning instead.

The recommendation is the *lowest* legal VFR cruising altitude at or above
the floor, per FAR 91.159 on the *magnetic* course (odd thousands +500 ft
for 000-179°, even +500 for 180-359°; the variation is the World Magnetic
Model's, `vfr.magnetic`). This used to round down from the ceiling on the
true course, which on a route with no low shelf recommended 12,500 ft for
a 100 nm leg across Iowa. `altitude.legal_cruising_altitudes` is the
planner's own list, so the last line checks it against the planner.

In [6]:
variation_deg = magnetic.magnetic_variation_deg(mid_lat, mid_lon)
course_magnetic_deg = (route_bearing_deg - variation_deg) % 360

ceilings = [c for c in [airspace_ceiling_ft, profile["service_ceiling_ft"]] if c is not None]
band_ceiling_ft = min(ceilings) if ceilings else None

legal = altitude.legal_cruising_altitudes(floor_ft, band_ceiling_ft, course_magnetic_deg)
recommended_ft = legal[0] if legal else None

print(f"Magnetic course {course_magnetic_deg:.0f} deg ({'eastbound' if altitude.is_eastbound(course_magnetic_deg) else 'westbound'})")
if recommended_ft is None:
    print(f"No valid VFR altitude: nothing legal between the floor {floor_ft:.0f} ft and the ceiling {band_ceiling_ft} ft")
else:
    print(f"Recommended cruising altitude: {recommended_ft:.0f} ft MSL")
    print(f"  (floor {floor_ft:.0f} ft, ceiling {band_ceiling_ft if band_ceiling_ft is not None else 'none'} ft)")

if (cv["min_ceiling_ft"] is not None and cv["min_ceiling_ft"] < 1000) or (cv["min_visibility_sm"] is not None and cv["min_visibility_sm"] < 3):
    print("GO/NO-GO: conditions near the route are below typical VFR minimums -- check a full briefing before flying")
if hazards:
    print(f"GO/NO-GO: {len(hazards)} SIGMET/AIRMET(s) intersect the route -- review before flying")

planner = altitude.select_cruise_altitude(route_start, route_end, profile, faa_cache_dir=FAA_CACHE_DIR)
print(f"The planner's own: {planner['recommended_ft']}")

Magnetic course 331 deg (westbound)
Recommended cruising altitude: 2500 ft MSL
  (floor 2200 ft, ceiling 3600.0 ft)
GO/NO-GO: conditions near the route are below typical VFR minimums -- check a full briefing before flying


The planner's own: 2500.0
